In [0]:
df = spark.sql("""
SELECT *
FROM policyprojcatalog.policyprojdb.customer
WHERE merge_flag = false
  AND customer_id IS NOT NULL
  AND gender IN ('Male', 'Female')
  AND registration_date > date_of_birth
""")

display(df)

In [0]:
df.createOrReplaceTempView("clean_customer")

spark.sql("""
MERGE INTO policyprojcatalog.silver.customer AS T
USING clean_customer AS S
ON T.customer_id = S.customer_id

WHEN MATCHED THEN UPDATE SET
  T.first_name = S.first_name,
  T.last_name = S.last_name,
  T.email = S.email,
  T.phone = S.phone,
  T.country = S.country,
  T.city = S.city,
  T.registration_date = S.registration_date,
  T.date_of_birth = S.date_of_birth,
  T.gender = S.gender,
  T.merged_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  customer_id,
  first_name,
  last_name,
  email,
  phone,
  country,
  city,
  registration_date,
  date_of_birth,
  gender,
  merged_timestamp
)
VALUES (
  S.customer_id,
  S.first_name,
  S.last_name,
  S.email,
  S.phone,
  S.country,
  S.city,
  S.registration_date,
  S.date_of_birth,
  S.gender,
  current_timestamp()
)
""")

In [0]:
%sql
UPDATE policyprojcatalog.policyprojdb.customer
SET merge_flag = true
WHERE merge_flag = false;